Convert text to speech and save it as a file

In [10]:
import os
import uuid
from dotenv import load_dotenv
from elevenlabs import VoiceSettings
from elevenlabs.client import ElevenLabs
from elevenlabs import play

load_dotenv()

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
elevenlabs = ElevenLabs(
  api_key=ELEVENLABS_API_KEY
)

def text_to_speech_file(text: str) -> str:
    # Calling tts API with params
    response = elevenlabs.text_to_speech.convert(
        voice_id="sDuUJMeNJR828mXTRrDh", # hernan cortés 
        output_format="mp3_22050_32",
        text=text,
        model_id="eleven_turbo_v2_5",
        voice_settings=VoiceSettings(
            stability=0.5,
            similarity_boost=1.0,
            style=0.0,
            use_speaker_boost=True,
            speed=1.0
        ),         # adam pre-made
    )

    play.play(response)

    # generate a unique filename for the output file
    save_file_path = f"{uuid.uuid4()}.mp3"

    # writing the audio to a file
    with open(save_file_path, "wb") as f:
        for chunk in response:
            if chunk:
                f.write(chunk)

    print(f"Audio saved to {save_file_path}")

    return save_file_path

In [13]:
print(text_to_speech_file("Hola bienaventurado!"))

Audio saved to 4e85e091-4291-4944-a5be-7054255a2198.mp3
4e85e091-4291-4944-a5be-7054255a2198.mp3


In [33]:
# same but streaming audio directly

import os 
from typing import IO
from io import BytesIO
from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs
from elevenlabs import play

load_dotenv()

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
elevenlabs = ElevenLabs(
    api_key=ELEVENLABS_API_KEY
)

def text_to_speech_stream(text: str) -> str:
    response = elevenlabs.text_to_speech.convert(
        voice_id="sDuUJMeNJR828mXTRrDh",
        output_format="mp3_22050_32",
        text=text,
        model_id="eleven_turbo_v2_5",
            voice_settings=VoiceSettings(
            stability=0.5, # how stable the voice is
            similarity_boost=1.0, # how closely the voice matches the text
            style=0.0, # style exaggeration. adds latency
            use_speaker_boost=True, # subtle differences to similar voices, adds latency
            speed=1 # should be between 0.7 and 1.2
        ),    
    )

    # create bytesIO object to hold the audio in memory
    audio_stream = BytesIO()

    # write the audio to the bytesIO object
    for chunk in response:
        if chunk:
            audio_stream.write(chunk)

    # reset the position of the bytesIO object to the beginning
    audio_stream.seek(0) # this is necessary to read the audio from the beginning
    # return the bytesIO object
    return audio_stream

In [30]:
text_to_speech_stream("Soy el capitán de los mares!")

In [35]:
audio_stream = text_to_speech_stream("Soy el capitán de los mares!")
play.play(audio_stream)